### Step 1 - Load Bronze data

In [0]:
silver_df = spark.table(
    "workspace.data_analysis.bronze_employee"
)

In [0]:
silver_df.show()

+-----------+----------+---------+----+-------------------+--------+----------+---------+--------------------+-----------+-----------------+-----------+---------+-------------+--------------------+
|Employee_ID|First_name|Last_name| Age|  Department_Region|  Status| Join_Date|   Salary|               Email|      Phone|Performance_Score|Remote_work| batch_id|source_system|    ingest_timestamp|
+-----------+----------+---------+----+-------------------+--------+----------+---------+--------------------+-----------+-----------------+-----------+---------+-------------+--------------------+
|    EMP1000|       Bob|    Davis|  25|  DevOps-California|  Active|  4/2/2021| 59767.65|bob.davis@example...|-1651623197|          Average|       true|BATCH_001| employee_csv|2026-09-25 13:54:...|
|    EMP1001|       Bob|    Brown|NULL|      Finance-Texas|  Active| 7/10/2020| 65304.66|bob.brown@example...|-1898471390|        Excellent|       true|BATCH_001| employee_csv|2026-09-25 13:54:...|
|    EMP10

In [0]:
silver_df.printSchema()
print("Total rows:",silver_df.count())

root
 |-- Employee_ID: string (nullable = true)
 |-- First_name: string (nullable = true)
 |-- Last_name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Department_Region: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- Join_Date: string (nullable = true)
 |-- Salary: double (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone: string (nullable = true)
 |-- Performance_Score: string (nullable = true)
 |-- Remote_work: boolean (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- ingest_timestamp: timestamp (nullable = true)

Total rows: 1020


### Step 2 - Find NULL Values

This should be our first actual analysis of the messy data.

We can check NULLs for every column:

In [0]:
from pyspark.sql.functions import sum,col

silver_df.select(
    [
    sum(col(column).isNull().cast("int")).alias(column)
    for column in silver_df.columns
]
).show()

+-----------+----------+---------+---+-----------------+------+---------+------+-----+-----+-----------------+-----------+--------+-------------+----------------+
|Employee_ID|First_name|Last_name|Age|Department_Region|Status|Join_Date|Salary|Email|Phone|Performance_Score|Remote_work|batch_id|source_system|ingest_timestamp|
+-----------+----------+---------+---+-----------------+------+---------+------+-----+-----+-----------------+-----------+--------+-------------+----------------+
|          0|         0|        0|211|                0|     0|        0|    24|    0|    0|                0|          0|       0|            0|               0|
+-----------+----------+---------+---+-----------------+------+---------+------+-----+-----+-----------------+-----------+--------+-------------+----------------+



In [0]:
department_list = ['CSE','AI&DS','AI&ML',None]

# Example 1
for department in department_list:
    print(department)

#Example 2
new_department_list = [department for department in department_list if department !=None]
print(new_department_list)

#Example 3
# Null department checking
for department in department_list:
    if department is None:
        print("NULL department found")

CSE
AI&DS
AI&ML
None
['CSE', 'AI&DS', 'AI&ML']
NULL department found


In [0]:
silver_df.select([
    col(c).isNull().cast("int").alias(c)
    for c in silver_df.columns
]
).show()

+-----------+----------+---------+---+-----------------+------+---------+------+-----+-----+-----------------+-----------+--------+-------------+----------------+
|Employee_ID|First_name|Last_name|Age|Department_Region|Status|Join_Date|Salary|Email|Phone|Performance_Score|Remote_work|batch_id|source_system|ingest_timestamp|
+-----------+----------+---------+---+-----------------+------+---------+------+-----+-----+-----------------+-----------+--------+-------------+----------------+
|          0|         0|        0|  0|                0|     0|        0|     0|    0|    0|                0|          0|       0|            0|               0|
|          0|         0|        0|  1|                0|     0|        0|     0|    0|    0|                0|          0|       0|            0|               0|
|          0|         0|        0|  1|                0|     0|        0|     0|    0|    0|                0|          0|       0|            0|               0|
|          0|         

### NULL + empty/whitespace values

In [0]:
from pyspark.sql.functions import trim
silver_df.select(
    [
        sum(
            (
            col(c).isNull() |
            (trim(col(c))== "")
            ).cast("int")
        ).alias(c)
        for c in silver_df.columns
    ]
).show()

+-----------+----------+---------+---+-----------------+------+---------+------+-----+-----+-----------------+-----------+--------+-------------+----------------+
|Employee_ID|First_name|Last_name|Age|Department_Region|Status|Join_Date|Salary|Email|Phone|Performance_Score|Remote_work|batch_id|source_system|ingest_timestamp|
+-----------+----------+---------+---+-----------------+------+---------+------+-----+-----+-----------------+-----------+--------+-------------+----------------+
|          0|         0|        0|211|                0|     0|        0|    24|    0|    0|                0|          0|       0|            0|               0|
+-----------+----------+---------+---+-----------------+------+---------+------+-----+-----+-----------------+-----------+--------+-------------+----------------+



### Step 3 - Clean and Standardize

### 3.1 First, categorize the columns
| Column              | Logical type     | What we need to do                    |
| ------------------- | ---------------- | ------------------------------------- |
| `Employee_ID`       | String           | Trim + validate                       |
| `First_name`        | String           | Trim + standardize                    |
| `Last_name`         | String           | Trim + standardize                    |
| `Age`               | Numeric          | Convert + validate                    |
| `Department_Region` | String           | Split into department + region        |
| `Status`            | Categorical      | Inspect distinct values + standardize |
| `Join_Date`         | Date             | Parse different date formats          |
| `Salary`            | Numeric          | Convert + validate                    |
| `Email`             | String           | Validate format                       |
| `Phone`             | String           | Validate format                       |
| `Performance_Score` | **Categorical**  | Inspect distinct values + validate    |
| `Remote_work`       | Boolean-like     | Inspect distinct values + standardize |
| Metadata            | String/timestamp | Preserve                              |


### Step 3.1 - Standardize basic string columns

In [0]:
from pyspark.sql.functions import col,trim,initcap

silver_df = (
    silver_df
    .withColumn("First_name",initcap(trim(col("First_name"))))
    .withColumn("Last_name",initcap(trim(col("Last_name"))))
)

### Step 3.2 - Understand categorical values
Profile first → understand the values → then define the cleaning rule.

### Performance Score

In [0]:
silver_df.select("Performance_Score").distinct().show()

+-----------------+
|Performance_Score|
+-----------------+
|          Average|
|        Excellent|
|             Good|
|             Poor|
+-----------------+



### Status

In [0]:
silver_df.select("Status").distinct().show()

+--------+
|  Status|
+--------+
|  Active|
| Pending|
|Inactive|
+--------+



### Remote Work

In [0]:
silver_df.select("Remote_work").distinct().show()

+-----------+
|Remote_work|
+-----------+
|       true|
|      false|
+-----------+



### Step 3.3 - Department and Region

In [0]:
from pyspark.sql.functions import split

silver_df = (
    silver_df
    .withColumn("Department",split(col("Department_Region"),'-').getItem(0))
    .withColumn("Region",split(col("Department_Region"),'-').getItem(1))
)

In [0]:
silver_df.select(
    "Department_Region",
    "Department",
    "Region"
).show()

+-------------------+----------+----------+
|  Department_Region|Department|    Region|
+-------------------+----------+----------+
|  DevOps-California|    DevOps|California|
|      Finance-Texas|   Finance|     Texas|
|       Admin-Nevada|     Admin|    Nevada|
|       Admin-Nevada|     Admin|    Nevada|
| Cloud Tech-Florida|Cloud Tech|   Florida|
|        Sales-Texas|     Sales|     Texas|
|       Admin-Nevada|     Admin|    Nevada|
| Cloud Tech-Florida|Cloud Tech|   Florida|
|       Admin-Nevada|     Admin|    Nevada|
|    DevOps-New York|    DevOps|  New York|
|Cloud Tech-New York|Cloud Tech|  New York|
|  DevOps-California|    DevOps|California|
|        HR-New York|        HR|  New York|
| Finance-California|   Finance|California|
|       Admin-Nevada|     Admin|    Nevada|
|      Sales-Florida|     Sales|   Florida|
|    DevOps-Illinois|    DevOps|  Illinois|
|      Finance-Texas|   Finance|     Texas|
|  Cloud Tech-Nevada|Cloud Tech|    Nevada|
|     Admin-Illinois|     Admin|

In [0]:
silver_df = silver_df.drop("Department_Region")

### Step 3.4 - Salary Validation

### Check for zero or negative salaries

In [0]:
silver_df.filter(col("salary")<=0).show()

+-----------+----------+---------+---+------+---------+------+-----+-----+-----------------+-----------+--------+-------------+----------------+----------+------+
|Employee_ID|First_name|Last_name|Age|Status|Join_Date|Salary|Email|Phone|Performance_Score|Remote_work|batch_id|source_system|ingest_timestamp|Department|Region|
+-----------+----------+---------+---+------+---------+------+-----+-----+-----------------+-----------+--------+-------------+----------------+----------+------+
+-----------+----------+---------+---+------+---------+------+-----+-----+-----------------+-----------+--------+-------------+----------------+----------+------+



### Check the salary range

In [0]:
silver_df.select("Salary").describe().show()

+-------+------------------+
|summary|            Salary|
+-------+------------------+
|  count|               996|
|   mean| 85155.05639558229|
| stddev|19873.727918135482|
|    min|          50047.32|
|    max|         119971.65|
+-------+------------------+



### Step 3.5 - Create Salary Validation Flag

In [0]:
silver_df = silver_df.withColumn(
    "is_salary_valid",
    col("Salary").isNotNull() & (col("Salary") > 0)
)

In [0]:
silver_df.show()

+-----------+----------+---------+----+--------+----------+---------+--------------------+-----------+-----------------+-----------+---------+-------------+--------------------+----------+----------+---------------+
|Employee_ID|First_name|Last_name| Age|  Status| Join_Date|   Salary|               Email|      Phone|Performance_Score|Remote_work| batch_id|source_system|    ingest_timestamp|Department|    Region|is_salary_valid|
+-----------+----------+---------+----+--------+----------+---------+--------------------+-----------+-----------------+-----------+---------+-------------+--------------------+----------+----------+---------------+
|    EMP1000|       Bob|    Davis|  25|  Active|  4/2/2021| 59767.65|bob.davis@example...|-1651623197|          Average|       true|BATCH_001| employee_csv|2026-09-25 13:54:...|    DevOps|California|           true|
|    EMP1001|       Bob|    Brown|NULL|  Active| 7/10/2020| 65304.66|bob.brown@example...|-1898471390|        Excellent|       true|BATC

In [0]:
my_age = 20
age = (my_age>=18) & (my_age<=70)
print(age)

True


### Step 3.6 - Age Validation & Cleaning

### 3.6.1 — Check invalid age values

In [0]:
silver_df.filter(
    ( col("Age")<0 ) | (col("Age") > 100)
).show()

+-----------+----------+---------+---+------+---------+------+-----+-----+-----------------+-----------+--------+-------------+----------------+----------+------+---------------+
|Employee_ID|First_name|Last_name|Age|Status|Join_Date|Salary|Email|Phone|Performance_Score|Remote_work|batch_id|source_system|ingest_timestamp|Department|Region|is_salary_valid|
+-----------+----------+---------+---+------+---------+------+-----+-----+-----------------+-----------+--------+-------------+----------------+----------+------+---------------+
+-----------+----------+---------+---+------+---------+------+-----+-----+-----------------+-----------+--------+-------------+----------------+----------+------+---------------+



### 3.6.2 — Handle missing Age

In [0]:
silver_df = silver_df.withColumn(
    "is_age_missing",
    col("Age").isNull()
)

In [0]:
silver_df.filter(col("is_age_missing")==True).count()

211

### 3.6.3 — Decide invalid Age records

In [0]:
# silver_df = silver_df.withColumn(
#     "is_age_valid",
#     (col("Age") >=0) & (col("Age")<=100)
# )

### Step 3.7 - Join_Date Standardization

### 3.7.1 - Parse the different formats

In [0]:
from pyspark.sql.functions import col,try_to_date,to_date,try_to_timestamp

# MM - month
# mm - minute

silver_df = silver_df.withColumn(
    "Join_Date",
    to_date(col("Join_Date"), "M/d/yyyy")
)

### 3.7.2 - Check whether any dates failed to parse

In [0]:
silver_df.filter(
    col('Join_Date').isNull()
).count()

0

### Step 3.8 - Employee_ID Validation

### 3.8.1 - Check the actual Employee_ID format

In [0]:
silver_df.select("Employee_ID").show(20,False)

+-----------+
|Employee_ID|
+-----------+
|EMP1000    |
|EMP1001    |
|EMP1002    |
|EMP1003    |
|EMP1004    |
|EMP1005    |
|EMP1006    |
|EMP1007    |
|EMP1008    |
|EMP1009    |
|EMP1010    |
|EMP1011    |
|EMP1012    |
|EMP1013    |
|EMP1014    |
|EMP1015    |
|EMP1016    |
|EMP1017    |
|EMP1018    |
|EMP1019    |
+-----------+
only showing top 20 rows


### 3.8.2 - Find invalid Employee_IDs

In [0]:
employee_pattern = r"^EMP[0-9]+$"
silver_df.filter(
    ~col("Employee_ID").rlike(employee_pattern)
).select("Employee_ID").show()

+-----------+
|Employee_ID|
+-----------+
+-----------+



### 3.8.3 - Check duplicate Employee_IDs

In [0]:
from pyspark.sql.functions import count

silver_df.groupBy("Employee_ID").agg(count("*").alias("count")).filter(col("count")>1).show()

+-----------+-----+
|Employee_ID|count|
+-----------+-----+
+-----------+-----+



### Step 3.9 - Email Validation

### 3.9.1 — Validate the email format

In [0]:
silver_df.filter(
    col("Email").rlike(r"^[a-zA-Z]+\.[a-zA-Z]+@example\.com$")
).select(
    "First_name",
    "Last_name",
    "Email"
).show()

+----------+---------+--------------------+
|First_name|Last_name|               Email|
+----------+---------+--------------------+
|       Bob|    Davis|bob.davis@example...|
|       Bob|    Brown|bob.brown@example...|
|     Alice|    Jones|alice.jones@examp...|
|       Eva|    Davis|eva.davis@example...|
|     Frank| Williams|frank.williams@ex...|
|     Alice|   Garcia|alice.garcia@exam...|
|     Frank|    Jones|frank.jones@examp...|
|       Bob|    Jones|bob.jones@example...|
|     Frank|    Davis|frank.davis@examp...|
|   Charlie|  Johnson|charlie.johnson@e...|
|     David| Williams|david.williams@ex...|
|     David|    Davis|david.davis@examp...|
|     Heidi| Williams|heidi.williams@ex...|
|     Grace| Williams|grace.williams@ex...|
|     Alice|   Garcia|alice.garcia@exam...|
|     Grace|    Jones|grace.jones@examp...|
|     Alice|    Brown|alice.brown@examp...|
|     Grace|   Garcia|grace.garcia@exam...|
|   Charlie|   Garcia|charlie.garcia@ex...|
|   Charlie| Williams|charlie.wi

### 3.9.2 - But our business rule is stronger

In [0]:
from pyspark.sql.functions import lower,concat,lit,col

silver_df.filter(
    lower(col("Email")) != 
    concat(
        lower(col("First_name")),
        lit("."),
        lower(col("Last_name")),
        lit("@example.com")
    )
).select(
    "First_name",
    "Last_name",
    "Email"
).show()

+----------+---------+-----+
|First_name|Last_name|Email|
+----------+---------+-----+
+----------+---------+-----+



### Step 3.10 - Phone Validation

### 3.10.1 - Check the actual data type

In [0]:
from pyspark.sql.functions import col

silver_df = silver_df.withColumn(
    "Phone",
    col("Phone").cast("string")
)

### 3.10.2 — Remove the leading -

In [0]:
from pyspark.sql.functions import regexp_replace

silver_df = silver_df.withColumn(
    "Phone",
    regexp_replace(col("Phone"),"^-","")
)

### 3.10.3 - Validate exactly 10 digits

In [0]:
silver_df.filter(
    ~col("Phone").rlike("^[0-9]{10}$")
).select("Phone").count()

92